In [1]:
import kagglehub
from confluent_kafka.admin import AdminClient, NewTopic
from pyspark.sql import SparkSession

try:
    spark.stop()
except:
    pass

spark = (SparkSession.builder.appName("MyApp")
    .config("spark.jars.packages",
        "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.3")    
    .master("local[*]")
    .getOrCreate())

/home/duyhuynh/Desktop/school/bigdata/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
26/03/25 16:11:39 WARN Utils: Your hostname, debian resolves to a loopback address: 127.0.1.1; using 10.130.201.180 instead (on interface wlo1)
26/03/25 16:11:39 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/duyhuynh/Desktop/school/bigdata/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/duyhuynh/.ivy2/cache
The jars for the packages stored in: /home/duyhuynh/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-4d29fbdc-00ae-4b5a-a3de-283b1c065b35;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.5.3 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.5.3 in central
	found org.apache.kafka#kafka-clients;3.4.1 in central
	found org.lz4#lz4-java;1.8.0 in local-m2-cache
	found org.xerial.snappy#snappy-java;1.1.10.5 in local-m2-cache
	found org.slf4j#slf4j-api;2.0.7 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.4 in central
	found commons-logging#commons-logging;1.1.3 in local-m2-cache
	found com.google.code.findbugs#jsr305;3.0.0 in local-m2-cache
	found org.apache.commons#commons-pool2;2.11.1 in central
downloading https://repo1.mave

In [2]:
path = kagglehub.dataset_download("grouplens/movielens-latest-small")
df_ratings = spark.read.csv(path + "/ratings.csv", header=True, inferSchema=True)
df_movies = spark.read.csv(path + "/movies.csv", header=True, inferSchema=True)
df_tags = spark.read.csv(path + "/tags.csv", header=True, inferSchema=True)
df_ratings.show(1)
df_movies.show(1)
df_tags.show(1)

+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|     1|      1|   4.0|964982703|
+------+-------+------+---------+
only showing top 1 row

+-------+----------------+--------------------+
|movieId|           title|              genres|
+-------+----------------+--------------------+
|      1|Toy Story (1995)|Adventure|Animati...|
+-------+----------------+--------------------+
only showing top 1 row

+------+-------+-----+----------+
|userId|movieId|  tag| timestamp|
+------+-------+-----+----------+
|     2|  60756|funny|1445714994|
+------+-------+-----+----------+
only showing top 1 row



In [3]:
bootstrap_servers = "localhost:30090,localhost:30091,localhost:30092"
# First, delete topics if they exist
admin_client = AdminClient({'bootstrap.servers': bootstrap_servers})
admin_client.delete_topics(['ratings', 'movies', 'tags'], operation_timeout=10)
# Create new topics
new_topics = [NewTopic(topic="ratings", num_partitions=1, replication_factor=3),
NewTopic(topic="movies", num_partitions=1, replication_factor=3),
NewTopic(topic="tags", num_partitions=1, replication_factor=3)]
fs = admin_client.create_topics(new_topics)
for topic, f in fs.items():
    try:
        f.result() # The result itself is None
        print(f"Topic {topic} created")
    except Exception as e:
        print(f"Failed to create topic {topic}: {e}")

Topic ratings created
Topic movies created
Topic tags created


In [ ]:
# Push the dataframes to kafka broker
df_ratings.selectExpr("to_json(struct(*)) AS value") \
.write \
.format("kafka") \
.option("kafka.bootstrap.servers", bootstrap_servers) \
.option("topic", "ratings") \
.save()

df_movies.selectExpr("to_json(struct(*)) AS value") \
.write \
.format("kafka") \
.option("kafka.bootstrap.servers", bootstrap_servers) \
.option("topic", "movies") \
.save()

df_tags.selectExpr("to_json(struct(*)) AS value") \
.write \
.format("kafka") \
.option("kafka.bootstrap.servers", bootstrap_servers) \
.option("topic", "tags") \
.save()

df_ratings_kafka = spark.read \
.format("kafka") \
.option("kafka.bootstrap.servers", bootstrap_servers) \
.option("subscribe", "ratings") \
.load()
df_ratings_kafka = df_ratings_kafka.selectExpr("CAST(value AS STRING)")
df_ratings_kafka.show(1)

26/03/25 16:13:05 WARN NetworkClient: [Producer clientId=producer-1] Error while fetching metadata with correlation id 1 : {ratings=UNKNOWN_TOPIC_OR_PARTITION}
26/03/25 16:13:05 WARN NetworkClient: [Producer clientId=producer-1] Error while fetching metadata with correlation id 4 : {ratings=UNKNOWN_TOPIC_OR_PARTITION}
26/03/25 16:13:05 WARN NetworkClient: [Producer clientId=producer-1] Error while fetching metadata with correlation id 5 : {ratings=UNKNOWN_TOPIC_OR_PARTITION}
26/03/25 16:13:07 WARN NetworkClient: [Producer clientId=producer-1] Error while fetching metadata with correlation id 1336 : {movies=UNKNOWN_TOPIC_OR_PARTITION}
26/03/25 16:13:07 WARN NetworkClient: [Producer clientId=producer-1] Error while fetching metadata with correlation id 1337 : {movies=UNKNOWN_TOPIC_OR_PARTITION}
26/03/25 16:13:07 WARN NetworkClient: [Producer clientId=producer-1] Error while fetching metadata with correlation id 1667 : {tags=UNKNOWN_TOPIC_OR_PARTITION}
26/03/25 16:13:07 WARN NetworkClient

+--------------------+
|               value|
+--------------------+
|{"userId":1,"movi...|
+--------------------+
only showing top 1 row

